# Лабораторна робота №5

Програма для обчислень за варіантом `15` з дисципліни **Теорія прийняття рішень**.

У ноутбуці реалізовано:
- побудову таблиць значень `I12(x, y)` та `I21(x, y)`;
- пошук гарантованого результату за принципами `max_x min_y` та `max_y min_x`;
- обчислення `FΣ12` і `FΣ21` з урахуванням факторів ризику;
- пошук найбільш несприятливої ситуації за ситуаційними матрицями ризику.


## Формули та вихідні дані

Область зміни змінних:

- `x ∈ [0, 2]`
- `y ∈ [0, 2]`
- крок сітки: `0.5`

Цільові функції:

$$I_{12}(x, y) = \frac{67 - 6x - 32y + 3x^2 + 4y^2}{7}$$

$$I_{21}(x, y) = \frac{26 - 7x - 20y + x^2 + 5y^2}{19}$$

Функції збитків для першої коаліції:

$$J_{12ns}(x, y) = \frac{35 + 5x - 7y + 3xy}{42}$$

$$J_{12fm}(x, y) = \frac{0.7 + 0.3x + 0.14y + 0.25xy}{3.1}$$

$$J_{12in}(x, y) = \frac{70y + 2x - 14y + 0.5xy}{35}$$

Функції збитків для другої коаліції:

$$J_{21ns}(x, y) = \frac{-35.8y + 1.2x - 1.4y + 5xy}{89.6}$$

$$J_{21fm}(x, y) = \frac{-10.94 + 1.32x - 0.4y + 0.57xy}{89.6}$$

$$J_{21in}(x, y) = \frac{-46y + x - 14y + 9xy}{136}$$

Узагальнені цільові функції з урахуванням ризику:

$$F_{\Sigma 12}(x, y) = (1 - \eta_{ns})(1 - \eta_{fm})(1 - \eta_{in})I_{12}(x, y) - (\eta_{ns}J_{12ns} + \eta_{fm}J_{12fm} + \eta_{in}J_{12in})$$

$$F_{\Sigma 21}(x, y) = (1 - \eta_{ns})(1 - \eta_{fm})(1 - \eta_{in})I_{21}(x, y) - (\eta_{ns}J_{21ns} + \eta_{fm}J_{21fm} + \eta_{in}J_{21in})$$


In [ ]:
from __future__ import annotations

import numpy as np

np.set_printoptions(suppress=True, precision=6)

STEP = 0.5
X_VALUES = np.arange(0.0, 2.0 + STEP / 2, STEP)
Y_VALUES = np.arange(0.0, 2.0 + STEP / 2, STEP)
SCENARIO_VALUES = np.array([0, 1, 2], dtype=float)

R1 = np.array([
    [0.10, 0.05, 0.20],
    [0.01, 0.10, 0.01],
    [0.12, 0.03, 0.06],
], dtype=float)

R2 = np.array([
    [0.20, 0.30, 0.12],
    [0.03, 0.09, 0.015],
    [0.07, 0.20, 0.075],
], dtype=float)


def I12(x: float, y: float) -> float:
    return (67 - 6 * x - 32 * y + 3 * x**2 + 4 * y**2) / 7


def I21(x: float, y: float) -> float:
    return (26 - 7 * x - 20 * y + x**2 + 5 * y**2) / 19


def J12ns(x: float, y: float) -> float:
    return (35 + 5 * x - 7 * y + 3 * x * y) / 42


def J12fm(x: float, y: float) -> float:
    return (0.7 + 0.3 * x + 0.14 * y + 0.25 * x * y) / 3.1


def J12in(x: float, y: float) -> float:
    return (70 * y + 2 * x - 14 * y + 0.5 * x * y) / 35


def J21ns(x: float, y: float) -> float:
    return (-35.8 * y + 1.2 * x - 1.4 * y + 5 * x * y) / 89.6


def J21fm(x: float, y: float) -> float:
    return (-10.94 + 1.32 * x - 0.4 * y + 0.57 * x * y) / 89.6


def J21in(x: float, y: float) -> float:
    return (-46 * y + x - 14 * y + 9 * x * y) / 136


def F12(x: float, y: float, etas: np.ndarray) -> float:
    eta_ns, eta_fm, eta_in = etas
    multiplier = (1 - eta_ns) * (1 - eta_fm) * (1 - eta_in)
    penalty = eta_ns * J12ns(x, y) + eta_fm * J12fm(x, y) + eta_in * J12in(x, y)
    return multiplier * I12(x, y) - penalty


def F21(x: float, y: float, etas: np.ndarray) -> float:
    eta_ns, eta_fm, eta_in = etas
    multiplier = (1 - eta_ns) * (1 - eta_fm) * (1 - eta_in)
    penalty = eta_ns * J21ns(x, y) + eta_fm * J21fm(x, y) + eta_in * J21in(x, y)
    return multiplier * I21(x, y) - penalty


def build_grid(func) -> np.ndarray:
    return np.array([[func(x, y) for y in Y_VALUES] for x in X_VALUES], dtype=float)


def print_value_table(grid: np.ndarray, title: str) -> None:
    print(title)
    print(f"{'#':>3} {'x':>4} {'y':>4} {'value':>12}")
    index = 0
    for i, x in enumerate(X_VALUES):
        for j, y in enumerate(Y_VALUES):
            print(f"{index:>3} {x:>4.1f} {y:>4.1f} {grid[i, j]:>12.6f}")
            index += 1


def scenario_index_from_value(value: float) -> int:
    rounded = int(round(value))
    if not np.isclose(value, rounded) or rounded not in (0, 1, 2):
        raise ValueError(f"Значення {value} не можна напряму зіставити зі сценаріями S1..S3")
    return rounded


def guaranteed_result_12(grid12: np.ndarray) -> dict:
    row_mins = grid12.min(axis=1)
    x_index = int(row_mins.argmax())
    y_index = int(grid12[x_index].argmin())
    return {
        'x': float(X_VALUES[x_index]),
        'y': float(Y_VALUES[y_index]),
        'value': float(grid12[x_index, y_index]),
        'row_mins': row_mins,
    }


def guaranteed_result_21(grid21: np.ndarray) -> dict:
    col_mins = grid21.min(axis=0)
    y_index = int(col_mins.argmax())
    x_index = int(grid21[:, y_index].argmin())
    return {
        'x': float(X_VALUES[x_index]),
        'y': float(Y_VALUES[y_index]),
        'value': float(grid21[x_index, y_index]),
        'col_mins': col_mins,
    }


def worst_scenario_probabilities(matrix: np.ndarray) -> np.ndarray:
    return np.prod(matrix, axis=0)


## Завдання 1. Гарантований результат

Для першої коаліції:

$$I^*_{12} = \max_x \min_y I_{12}(x, y)$$

Для другої коаліції:

$$I^*_{21} = \max_y \min_x I_{21}(x, y)$$


In [ ]:
grid12 = build_grid(I12)
grid21 = build_grid(I21)

print_value_table(grid12, 'Таблиця значень I12(x, y)')
print()
print_value_table(grid21, 'Таблиця значень I21(x, y)')

guarantee_12 = guaranteed_result_12(grid12)
guarantee_21 = guaranteed_result_21(grid21)

print()
print('Мінімальні значення I12 по y для кожного x:')
for x, value in zip(X_VALUES, guarantee_12['row_mins']):
    print(f'x = {x:>3.1f} -> min_y I12 = {value:.6f}')

print()
print('Мінімальні значення I21 по x для кожного y:')
for y, value in zip(Y_VALUES, guarantee_21['col_mins']):
    print(f'y = {y:>3.1f} -> min_x I21 = {value:.6f}')

print()
print('Гарантований результат для 1-ї коаліції:')
print(guarantee_12)

print()
print('Гарантований результат для 2-ї коаліції:')
print(guarantee_21)


## Завдання 2. Значення цільових функцій з урахуванням ризику

Для першої коаліції беремо стовпець `R1`, що відповідає значенню `y*`.

Для другої коаліції беремо стовпець `R2`, що відповідає значенню `x*`.


In [ ]:
eta_12 = R1[:, scenario_index_from_value(guarantee_12['y'])]
eta_21 = R2[:, scenario_index_from_value(guarantee_21['x'])]

f12_guaranteed = F12(guarantee_12['x'], guarantee_12['y'], eta_12)
f21_guaranteed = F21(guarantee_21['x'], guarantee_21['y'], eta_21)

print('eta для 1-ї коаліції:', eta_12)
print('eta для 2-ї коаліції:', eta_21)

print()
print(f"FΣ12({guarantee_12['x']}, {guarantee_12['y']}) = {f12_guaranteed:.12f}")
print(f"FΣ21({guarantee_21['x']}, {guarantee_21['y']}) = {f21_guaranteed:.12f}")


## Завдання 3. Найбільш несприятлива ситуація

Для кожної коаліції:

1. обчислюємо ймовірність кожної ситуації як добуток трьох ймовірностей у відповідному стовпці матриці;
2. вибираємо стовпець із найбільшим добутком;
3. фіксуємо відповідне значення `y` для `R1` або `x` для `R2`;
4. знаходимо найбільше значення цільової функції за другою змінною;
5. окремо обчислюємо значення з урахуванням ризику.


In [ ]:
scenario_probs_1 = worst_scenario_probabilities(R1)
scenario_probs_2 = worst_scenario_probabilities(R2)

worst_index_1 = int(scenario_probs_1.argmax())
worst_index_2 = int(scenario_probs_2.argmax())

fixed_y = float(SCENARIO_VALUES[worst_index_1])
fixed_x = float(SCENARIO_VALUES[worst_index_2])

y_grid_index = int(np.where(np.isclose(Y_VALUES, fixed_y))[0][0])
x_grid_index = int(np.where(np.isclose(X_VALUES, fixed_x))[0][0])

best_x_for_worst_1 = int(grid12[:, y_grid_index].argmax())
best_y_for_worst_2 = int(grid21[x_grid_index, :].argmax())

worst_result_12 = {
    'scenario': f'S{worst_index_1 + 1}',
    'x': float(X_VALUES[best_x_for_worst_1]),
    'y': fixed_y,
    'value': float(grid12[best_x_for_worst_1, y_grid_index]),
    'probability': float(scenario_probs_1[worst_index_1]),
    'f_sigma': float(F12(X_VALUES[best_x_for_worst_1], fixed_y, R1[:, worst_index_1])),
}

worst_result_21 = {
    'scenario': f'S{worst_index_2 + 1}',
    'x': fixed_x,
    'y': float(Y_VALUES[best_y_for_worst_2]),
    'value': float(grid21[x_grid_index, best_y_for_worst_2]),
    'probability': float(scenario_probs_2[worst_index_2]),
    'f_sigma': float(F21(fixed_x, Y_VALUES[best_y_for_worst_2], R2[:, worst_index_2])),
}

print('Ймовірності ситуацій для 1-ї коаліції:', scenario_probs_1)
print('Ймовірності ситуацій для 2-ї коаліції:', scenario_probs_2)

print()
print('Найбільш несприятлива ситуація для 1-ї коаліції:')
print(worst_result_12)

print()
print('Найбільш несприятлива ситуація для 2-ї коаліції:')
print(worst_result_21)


## Підсумкова перевірка

Контрольні значення для перевірки обчислень:

- `I12* = 2.714285714286`
- `I21* = 0.842105263158`
- `FΣ12(0, 2) = 1.725570138249`
- `FΣ21(2, 0) = 0.672263860985`
- `I12_worst = 5.571428571429`
- `FΣ12_worst = 4.512234178187`
- `I21_worst = 1.052631578947`
- `FΣ21_worst = 0.540595553682`


In [ ]:
assert np.isclose(guarantee_12['value'], 2.714285714286)
assert np.isclose(guarantee_21['value'], 0.842105263158)
assert np.isclose(f12_guaranteed, 1.725570138249)
assert np.isclose(f21_guaranteed, 0.672263860985)
assert np.isclose(worst_result_12['value'], 5.571428571429)
assert np.isclose(worst_result_12['f_sigma'], 4.512234178187)
assert np.isclose(worst_result_21['value'], 1.052631578947)
assert np.isclose(worst_result_21['f_sigma'], 0.540595553682)

print('Контрольні перевірки пройдено.')
